# 🧠 Intelligible Blackboard Architecture — Live GPU Deliberation

This Google Colab notebook runs real multi-agent debates with local GPU models (`Qwen2.5-7B`), demonstrates autonomous deadlock detection and counterfactual sandbox recovery, executes live ablation benchmarks, and renders an interactive animated flowchart visualizer.

> **Requirement:** Go to `Runtime` -> `Change runtime type` -> Select **T4 GPU**.

## 1. Verify GPU Allocation

In [ ]:
!nvidia-smi

## 2. Clone Public Repository & Install Dependencies

In [ ]:
# Clone prototype branch and install dependencies
!git clone -b prototype-live-demo https://github.com/nikhil0744/intelligible-blackboard-architecture.git /content/project_blackboard
%cd /content/project_blackboard
%pip install -q -e ".[dev]"

## 3. Install & Start Ollama Server on Colab GPU

In [ ]:
# 1. Install zstd (required by Ollama for extraction) and Ollama binary
!apt-get update -qq && apt-get install -y -qq zstd pciutils > /dev/null
!curl -fsSL https://ollama.com/install.sh | sh

# 2. Launch Ollama daemon in background
import subprocess, time, urllib.request, os

env = dict(os.environ, OLLAMA_HOST="127.0.0.1:11434")
server_log = open("/content/ollama.log", "w")
ollama_proc = subprocess.Popen(["ollama", "serve"], env=env, stdout=server_log, stderr=subprocess.STDOUT)

print("Waiting for Ollama daemon to initialize...")
for _ in range(30):
    try:
        urllib.request.urlopen("http://127.0.0.1:11434/api/version", timeout=2)
        print("✅ Ollama server is running and ready!")
        break
    except Exception:
        time.sleep(1)
else:
    print("❌ Ollama server failed to start. Log output:")
    server_log.close()
    print(open("/content/ollama.log").read())

## 4. Download Model (Qwen2.5-7B-Instruct)

Downloads in ~1 to 2 minutes on Colab's high-speed connection.

In [ ]:
MODEL = "qwen2.5:7b-instruct-q4_K_M"
!ollama pull {MODEL}

## 5. Live Demo 1: Multi-Agent Deliberation with Real LLM

Watch 3 specialized agents debate a complex clinical case on the shared blackboard using real GPU inference, PXP tags, confidence ratings, and token accounting.

In [ ]:
# Real 4-turn medical debate on GPU
!python -m scripts.s2_smoke --backend ollama --model qwen2.5:7b-instruct-q4_K_M --domain medical --turns 4 --parallel

## 6. Live Demo 2: Deadlock Detection & Counterfactual Sandbox Recovery

Demonstrates how the system catches conflicting agents in a deadlock loop, rolls back time into an isolated simulation sandbox, discovers an alternative hypothesis, and injects the resolution back to the live board.

In [ ]:
!python -m fixtures.spike_starter

## 7. Live Demo 3: Multi-Domain Benchmark & Ablation Study

Executes ablation trials across 4 counterfactual density conditions (0%, 33%, 66%, 100%) and exports research results.

In [ ]:
# Run 4 live ablation trials using the Qwen model on GPU
!python -m benchmarks.mscore --live --backend ollama --model qwen2.5:7b-instruct-q4_K_M --trials 4

## 8. View Generated Research Figures

Display the publication plots generated directly in `outputs/figures/`.

In [ ]:
from IPython.display import Image, display
import os

figures = [
    ("Consensus vs Counterfactual Density", "outputs/figures/fig1_consensus_vs_density.png"),
    ("Token Cost vs Recovery Rate", "outputs/figures/fig2_cost_vs_recovery.png"),
    ("Intelligibility Progression", "outputs/figures/fig3_intelligibility_progression.png"),
]

for title, path in figures:
    if os.path.exists(path):
        print(f"\n=== {title} ===")
        display(Image(filename=path))
    else:
        print(f"Figure not found at {path}")

## 9. Interactive Animated Flowchart Visualizer (Directly in Colab)

Run this cell to render the interactive multi-agent visualizer! It **automatically detects and loads your actual live execution trace** from whichever demo you just ran above, displaying the exact thoughts, rationales, token costs, and sandbox states of your agents:
- **🔴 Live Run Trace**: Ingests `outputs/live_trace.json` with the exact LLM thoughts and rationales from your latest run.
- **Scenario 1**: Deadlock & Sandbox Recovery (Demo 2 / Fixture Spike — Clinical IPF vs CHF)
- **Scenario 2**: Real Colab GPU Run (Demo 1 / s2_smoke — Metformin eGFR 28)
- **Scenario 3**: Live Benchmark Ablation (Demo 3 / MSCoRe Multi-Density Deadlock Recovery across 0%, 33%, 66%, 100%)

In [ ]:
import os, glob, json
import IPython.display as display

# Locate visualizer.html or fetch directly from GitHub
matches = glob.glob("/content/**/visualizer.html", recursive=True)
if matches and os.path.exists(matches[0]):
    target_path = matches[0]
else:
    !curl -s -O https://raw.githubusercontent.com/nikhil0744/intelligible-blackboard-architecture/prototype-live-demo/visualizer.html
    target_path = "visualizer.html"

with open(target_path, "r", encoding="utf-8") as f:
    html_content = f.read()

# Automatically inject live execution trace if generated by Demo 1, 2, or 3
trace_candidates = glob.glob("/content/**/outputs/live_trace.json", recursive=True) + glob.glob("outputs/live_trace.json")
valid_traces = [p for p in trace_candidates if os.path.exists(p) and os.path.getsize(p) > 20]
if valid_traces:
    trace_path = valid_traces[0]
    with open(trace_path, "r", encoding="utf-8") as tf:
        trace_json_str = tf.read()
    print(f"[*] Injected live execution trace from: {trace_path}")
    # Inject window.LIVE_TRACE_DATA before script execution
    injection = f"<script>window.LIVE_TRACE_DATA = {trace_json_str};</script>"
    html_content = html_content.replace("<head>", f"<head>\n  {injection}")
else:
    print("[*] No live_trace.json detected yet. Showing calibrated scenarios (run Cell 5, 6, or 7 to capture live traces).")

# Display directly inside Colab
display.display(display.HTML(html_content))


## 10. Cleanup & Shutdown

In [ ]:
ollama_proc.terminate()
server_log.close()
print("Ollama daemon stopped.")